In [3]:
%pip -q install "qiskit==2.2.3" "qiskit-algorithms==0.4.0" "qiskit-optimization==0.7.0" "koreanize-matplotlib==0.1.1"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [12]:
from itertools import product

import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Statevector
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

# 1. 입력 데이터
disaster = np.array([3, 2, 4, 3, 5, 4, 2, 2, 4, 4])
sites = [
    "인천 서구", "평택 고덕", "대구 성서", "광주 하남", "군산 산업단지",
    "포항 영일만", "부산 강서", "천안 직산", "원주 지정면", "울산 온산"
]
land_cost = np.array([320, 410, 280, 270, 250, 290, 450, 340, 230, 370])
logistics_km = np.array([45, 65, 230, 290, 180, 220, 80, 95, 160, 210])
infra_grade = np.array([5, 5, 4, 3, 3, 4, 5, 4, 3, 5])
labor_index = np.array([1.2, 1.1, 0.9, 0.85, 0.8, 0.9, 1.3, 1.0, 0.95, 1.1])
env_risk = np.array([15, 25, 40, 35, 30, 20, 20, 10, 25, 30])
permit_risk = np.array([20, 30, 15, 25, 40, 20, 35, 10, 25, 45])

# 2. 업무 지표 계산
risk = 0.4*env_risk+0.4*permit_risk+3.2*disaster
cost = land_cost+0.5*logistics_km+100*labor_index
objective = 0.8*cost+0.2*risk

# 3. 이진 변수, 목적함수, 제약조건
problem = QuadraticProgram()
for name in sites:
    problem.binary_var(name)
problem.minimize(linear=objective)
problem.linear_constraint({name: 1 for name in sites}, '==', 3, 'three_sites')

# 4. QUBO와 Ising 해밀토니안
converter = QuadraticProgramToQubo()
qubo = converter.convert(problem)
hamiltonian, offset = qubo.to_ising()
num_qubits = qubo.get_num_vars()
print("원래 변수 수 / 회로 큐비트 수:", problem.get_num_vars(), num_qubits)

# 5. 표준 QAOA 회로
layers = 1
gamma = ParameterVector("gamma", layers)  # 감마, 비용 각도
beta = ParameterVector("beta", layers)  # 베타, 믹서 각도
circuit = QuantumCircuit(num_qubits)
circuit.h(range(num_qubits))
for layer in range(layers):
    # I, Z, ZZ 비용 항은 서로 교환합니다.
    circuit.append(PauliEvolutionGate(hamiltonian, time=gamma[layer]), range(num_qubits))
    circuit.rx(2 * beta[layer], range(num_qubits))

# 6. 기대 에너지 계산과 각도 최적화
def state_at(angles):
    bindings = dict(zip([*gamma, *beta], angles))
    return Statevector.from_instruction(circuit.assign_parameters(bindings))

def expected_energy(angles):
    return float(state_at(angles).expectation_value(hamiltonian).real)

initial_angles = np.concatenate((np.full(layers, 0.5), np.full(layers, 0.8)))
optimization = minimize(expected_energy, initial_angles, method="COBYLA",
                        options={"maxiter": 70})
print("최종 감마·베타:", optimization.x)
print("최종 Ising 기대 에너지:", optimization.fun)
print("최종 QUBO 기대 비용:", optimization.fun + offset)
print("최적화 상태:", optimization.message)

# 7. 최종 상태 측정
shots = 2048
final_state = state_at(optimization.x)
final_state.seed(42)
counts = final_state.sample_counts(shots)
assert sum(counts.values()) == shots

# 8. 원래 변수 복원과 업무 결과 검증
exact_value = min(
    float(problem.objective.evaluate(np.asarray(bits)))
    for bits in product((0, 1), repeat=problem.get_num_vars())
    if problem.is_feasible(np.asarray(bits))
)
feasible_samples = []
for bitstring, count in counts.items():
    qubo_bits = np.array([int(bit) for bit in reversed(bitstring)])
    bits = np.asarray(converter.interpret(qubo_bits), dtype=int)
    if problem.is_feasible(bits):
        value = float(problem.objective.evaluate(bits))
        feasible_samples.append((value, tuple(bits), count))

feasible_count = sum(count for _, _, count in feasible_samples)
optimal_count = sum(count for value, _, count in feasible_samples
                    if np.isclose(value, exact_value, rtol=0, atol=1e-8))
print("실행 가능 해 / 최적해 관측 비율:", feasible_count / shots, optimal_count / shots)
if not feasible_samples:
    raise RuntimeError("실행 가능 해의 관측 횟수는 0입니다.")

selected_value, selected_bits, _ = min(feasible_samples)
chosen = np.array(selected_bits)
print("선택 비트:", chosen)
print("선택 항목:", [var.name for var, bit in zip(problem.variables, chosen) if bit])
print("원래 목적값 / 정확 최솟값 / 차이:",
      selected_value, exact_value, selected_value - exact_value)
for constraint in problem.linear_constraints:
    lhs = float(constraint.linear.evaluate(chosen))
    rhs = constraint.rhs
    equal = np.isclose(lhs, rhs)
    checks = {"EQ": equal, "LE": lhs <= rhs or equal, "GE": lhs >= rhs or equal}
    print(constraint.name, "계산값:", lhs, "기준:", rhs,
          "충족:", bool(checks[constraint.sense.name]))
assert problem.is_feasible(chosen)

# 9. 결과 저장과 업무 지표 확인
np.savez('sites_result.npz', names=np.array(sites), chosen=chosen, objective=selected_value,
         infrastructure=infra_grade)
print('선택 부지:', [name for name, b in zip(sites, chosen) if b > 0.5])

from itertools import combinations
site_comparison = []
for i, name in enumerate(sites):
    print(name, f'비용 점수={cost[i]:.2f}, 위험={risk[i]:.2f}, 최종 점수={objective[i]:.2f}')
for group in combinations(range(len(sites)), 3):
    site_comparison.append((float(objective[list(group)].sum()), group))
for value, group in sorted(site_comparison)[:5]:
    print('상위 비교 조합:', [sites[i] for i in group], '점수:', round(value, 2))

eligible = np.flatnonzero(infra_grade >= 4)
restricted = [(float(objective[list(g)].sum()), g) for g in combinations(eligible, 3)]
value, group = min(restricted, key=lambda item: item[0])
print('인프라 4 이상 조건의 정확 비교:', [sites[i] for i in group], round(value, 2))

원래 변수 수 / 회로 큐비트 수: 10 10
최종 감마·베타: [0.23621109 0.77110416]
최종 Ising 기대 에너지: -23017.36791784319
최종 QUBO 기대 비용: 5467.372082156908
최적화 상태: Return from COBYLA because the trust region radius reaches its lower bound.
실행 가능 해 / 최적해 관측 비율: 0.423828125 0.00048828125
선택 비트: [1 0 0 0 1 0 0 0 1 0]
선택 항목: ['인천 서구', '군산 산업단지', '원주 지정면']
원래 목적값 / 정확 최솟값 / 차이: 1050.08 1050.08 0.0
three_sites 계산값: 3.0 기준: 3 충족: True
선택 부지: ['인천 서구', '군산 산업단지', '원주 지정면']
인천 서구 비용 점수=462.50, 위험=23.60, 최종 점수=374.72
평택 고덕 비용 점수=552.50, 위험=28.40, 최종 점수=447.68
대구 성서 비용 점수=485.00, 위험=34.80, 최종 점수=394.96
광주 하남 비용 점수=500.00, 위험=33.60, 최종 점수=406.72
군산 산업단지 비용 점수=420.00, 위험=44.00, 최종 점수=344.80
포항 영일만 비용 점수=490.00, 위험=28.80, 최종 점수=397.76
부산 강서 비용 점수=620.00, 위험=28.40, 최종 점수=501.68
천안 직산 비용 점수=487.50, 위험=14.40, 최종 점수=392.88
원주 지정면 비용 점수=405.00, 위험=32.80, 최종 점수=330.56
울산 온산 비용 점수=585.00, 위험=42.80, 최종 점수=476.56
상위 비교 조합: ['인천 서구', '군산 산업단지', '원주 지정면'] 점수: 1050.08
상위 비교 조합: ['군산 산업단지', '천안 직산', '원주 지정면'] 점수: 1068.24
상위 비교 조합: ['대구 성서